In [ ]:
import os
import numpy as np
import ants
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output

# --- 1. CONFIGURATION ---
current_dataset = "ISBI"  # Change this to MICCAI or MSLegSeg as needed

index = {
    "ISBI": {
        "FLAIR": "/home/darshan/MS/data/PREPROCESSED/S27/flair.nii.gz",
        "T1": "/home/darshan/MS/data/PREPROCESSED/S27/t1.nii.gz",
        "PRED_MASK": "/home/darshan/MS/models/unet/results/adam_1e-5_8_1/pred_mask.nii.gz",
        "MASK": "/home/darshan/MS/data/PREPROCESSED/S27/mask.nii.gz"
    },
    # ... (Add your other datasets here if needed)
}

def load_volume_data(dataset_name):
    """Loads all 4 modalities for the dataset into numpy arrays."""
    paths = index.get(dataset_name)
    if not paths:
        return None, 0
    
    data_store = {}
    z_dims = []
    
    print(f"Loading {dataset_name} volumes...")
    for mod, path in paths.items():
        if os.path.exists(path):
            # Load with ANTs and convert to numpy immediately
            img = ants.image_read(path)
            data_store[mod] = img.numpy()
            z_dims.append(img.shape[2])
        else:
            data_store[mod] = None
    
    # Calculate safe slice range
    min_z = min(z_dims) if z_dims else 0
    return data_store, min_z

# --- 2. CREATE ANIMATION ---
def create_html_viewer(dataset_name, step=2):
    """
    Generates a portable HTML player.
    step=2: Skips every other slice to make generation faster. 
    """
    volumes, max_z = load_volume_data(dataset_name)
    if max_z == 0:
        print("Error: No data found.")
        return

    # Set up the static 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    fig.suptitle(f"{dataset_name} Interactive View", fontsize=14, fontweight='bold')
    
    # Flatten axes for easy iteration
    ax_list = {
        "FLAIR": axes[0,0], "T1": axes[0,1],
        "PRED_MASK": axes[1,0],    "MASK": axes[1,1]
    }
    
    # Initialize plots with empty data
    img_objs = {}
    for mod, ax in ax_list.items():
        vol = volumes[mod]
        if vol is not None:
            # Display middle slice initially
            im = ax.imshow(vol[:, :, max_z//2].T, cmap='gray', origin='lower', animated=True)
            ax.set_title(mod)
            ax.axis('off')
            img_objs[mod] = im
        else:
            ax.text(0.5, 0.5, "Missing", ha='center')
            ax.axis('off')

    plt.close(fig) # Prevent static plot from showing up separately

    # Update function for the animation loop
    def update(frame_idx):
        for mod, im in img_objs.items():
            # Update the image data for the new slice
            im.set_data(volumes[mod][:, :, frame_idx].T)
            # Optional: Update title with slice number (can be slow, maybe skip)
            # ax_list[mod].set_title(f"{mod} (z={frame_idx})")
        return list(img_objs.values())

    # Generate frames (using 'step' to reduce wait time)
    frames = range(0, max_z, step)
    print(f"Generating animation for {len(frames)} slices. Please wait...")
    
    anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)
    
    # Render to JavaScript HTML
    return HTML(anim.to_jshtml())

# --- 3. RUN ---
# This will take 10-20 seconds to render, but will be perfectly smooth afterwards
html_view = create_html_viewer(current_dataset, step=2)
display(html_view)


Loading ISBI volumes...
Generating animation for 91 slices. Please wait...
